# INTERACTIVE / DIAGNOSTIC
**NOT RECOMMENDED FOR DURABLE TRAINING.** Use the committed train/evaluate notebooks through Kaggle Save Version → Save & Run All.

In [ ]:
# 1 — Isolate one GPU before importing torch or training code
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['PLANNERAGENT_EXECUTION_SURFACE']='KAGGLE'

In [ ]:
# 2 — Environment and single-GPU audit
import platform, subprocess, sys
subprocess.run(['nvidia-smi'],check=True)
import torch
assert torch.cuda.is_available(),'KAGGLE_CUDA_GPU_REQUIRED'
assert torch.cuda.device_count()==1,'GCC4K_REQUIRES_EXACTLY_ONE_VISIBLE_GPU'
props=torch.cuda.get_device_properties(0)
print({'execution_surface':'KAGGLE','platform':platform.platform(),'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'visible_gpu_count':torch.cuda.device_count(),'vram_bytes':props.total_memory})

In [ ]:
# 3 — Pinned dependencies and torchao cleanup
import importlib.util, subprocess, sys
if importlib.util.find_spec('torchao') is not None: subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.51.3','peft==0.20.0','accelerate','safetensors','psutil'],check=True)
os.environ['WANDB_DISABLED']='true';os.environ['HF_HUB_DISABLE_TELEMETRY']='1'

In [ ]:
# 4 — Establish configurable Kaggle runtime persistence root and provenance
from pathlib import Path
import json, peft, transformers
PERSIST_ROOT=os.getenv('PLANNERAGENT_GCC4K_PERSIST_ROOT','/kaggle/working/PlannerAgent/GCC4K')
os.environ['PLANNERAGENT_GCC4K_PERSIST_ROOT']=PERSIST_ROOT
candidate=Path(PERSIST_ROOT)/'PA-INTERPRETATION-STUDENT-v0.2';state=candidate/'state';state.mkdir(parents=True,exist_ok=True)
kaggle_environment={'execution_surface':'KAGGLE','platform':platform.platform(),'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'visible_gpu_count':torch.cuda.device_count(),'vram_bytes':props.total_memory,'transformers':transformers.__version__,'peft':peft.__version__}
(state/'kaggle.environment.json').write_text(json.dumps(kaggle_environment,sort_keys=True,indent=2)+'\n')
print('RUNTIME_PERSISTED',PERSIST_ROOT)
print('KAGGLE_OUTPUT_PRESERVED = false until Save Version preserves /kaggle/working output')

In [ ]:
# 5 — Locate and extract the existing GCC4K recovery input bundle
import shutil
matches=list(Path('/kaggle/input').rglob('PA-INTERPRETATION-STUDENT-v0.2-GCC4K-RECOVERY-INPUT.zip'))
assert len(matches)==1,f'Expected one recovery input bundle, found {len(matches)}'
BUNDLE=matches[0];ROOT=Path('/kaggle/working/gcc4k');shutil.rmtree(ROOT,ignore_errors=True);shutil.unpack_archive(BUNDLE,ROOT)
print(BUNDLE)

In [ ]:
# 6 — Verify recovery bundle SHA-256, payload hashes, and POSIX paths
import hashlib, json, zipfile
EXPECTED_BUNDLE_SHA256='ab13d4b3bc7c6d19b8f83f0fc5e764687dde8acdc29239ff8d4782750e80205c'
EXPECTED_CORPUS_DIGEST='sha256:bd1b19fc6733cca6622051e30ac03dc3cbac3a602997894c7b4a1733467f3619'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()==EXPECTED_BUNDLE_SHA256,'GCC4K_RECOVERY_INPUT_SHA256_MISMATCH'
with zipfile.ZipFile(BUNDLE) as archive: assert all('\\' not in name and not name.startswith('/') and '..' not in Path(name).parts for name in archive.namelist()),'NON_POSIX_OR_UNSAFE_ZIP_PATH'
for line in (ROOT/'SHA256SUMS.txt').read_text().splitlines():
    expected,relative=line.split('  ',1);assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()==expected,relative
corpus_manifest=json.loads((ROOT/'corpus/manifest.json').read_text());assert corpus_manifest['aggregate_corpus_digest']==EXPECTED_CORPUS_DIGEST,'GCC4K_CORPUS_DIGEST_MISMATCH'
print('GCC4K RECOVERY INPUT VERIFIED',EXPECTED_BUNDLE_SHA256,EXPECTED_CORPUS_DIGEST)

In [ ]:
# 7 — Compatibility and exact pinned-base Internet preflight
import inspect, peft, transformers
from huggingface_hub import HfApi
from transformers import Trainer, TrainingArguments
assert transformers.__version__=='4.51.3';assert peft.__version__=='0.20.0'
assert 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters;assert 'eval_dataset' in inspect.signature(Trainer.__init__).parameters
try: HfApi().model_info('Qwen/Qwen3-0.6B-Base',revision='da87bfb608c14b7cf20ba1ce41287e8de496c0cd')
except Exception as error: raise RuntimeError(f'KAGGLE_INTERNET_OR_PINNED_BASE_UNAVAILABLE: {error}') from error
subprocess.run([sys.executable,str(ROOT/'scripts/train_targeted_student_v02.py'),'--help'],check=True)
print('COMPATIBILITY AND PINNED BASE PREFLIGHT PASS')

In [ ]:
# 8 — Show recovery state; shared gcc4k_recovery.py remains authoritative
for name in ('state','checkpoints','final-adapter','evaluation','result'):
    path=candidate/name;print(name,'exists' if path.exists() else 'missing')

In [ ]:
# 9 — TRAIN phase only: fresh, checkpoint resume, or verified adapter skip
subprocess.run([sys.executable,str(ROOT/'scripts/train_targeted_student_v02.py'),'--phase','train','--persist-root',PERSIST_ROOT],check=True)

In [ ]:
# 10 — Verify and expose adapter ZIP before evaluation
import json
final_adapter=candidate/'final-adapter';assert (final_adapter/'COMPLETE').is_file()
manifest=json.loads((final_adapter/'candidate.manifest.json').read_text())
for relative,expected in manifest['artifact_hashes'].items(): assert hashlib.sha256((final_adapter/relative).read_bytes()).hexdigest()==expected,relative
adapter_zip=candidate/'result'/'PA-INTERPRETATION-STUDENT-v0.2-ADAPTER-GCC4K.zip';adapter_zip_manifest=json.loads((candidate/'result'/'adapter-zip.manifest.json').read_text())
adapter_sha=hashlib.sha256(adapter_zip.read_bytes()).hexdigest();assert adapter_sha==adapter_zip_manifest['sha256']
print('ADAPTER_READY_FOR_KAGGLE_OUTPUT_PRESERVATION');print({'path':str(adapter_zip),'size':adapter_zip.stat().st_size,'sha256':adapter_sha})

In [ ]:
# 11 — EVALUATE phase only; reuses verified per-dataset results
subprocess.run([sys.executable,str(ROOT/'scripts/train_targeted_student_v02.py'),'--phase','evaluate','--persist-root',PERSIST_ROOT],check=True)

In [ ]:
# 12 — Verify and expose final result ZIP after evaluation
result_zip=candidate/'result'/'PA-INTERPRETATION-STUDENT-v0.2-GCC4K.zip';result_manifest=json.loads((candidate/'result'/'final-result.manifest.json').read_text())
assert (candidate/'result'/'FINAL_RESULT_COMPLETE').is_file();result_sha=hashlib.sha256(result_zip.read_bytes()).hexdigest();assert result_sha==result_manifest['zip_sha256']
with zipfile.ZipFile(result_zip) as archive: assert archive.testzip() is None
print('FINAL_RESULT_READY_FOR_KAGGLE_OUTPUT_PRESERVATION');print({'path':str(result_zip),'size':result_zip.stat().st_size,'sha256':result_sha})

In [ ]:
# 13 — Exact Kaggle outputs to preserve with Save Version
print('RUNTIME_PERSISTED artifacts are not cross-session durable until Kaggle Save Version succeeds.')
print('SAVE_AS_KAGGLE_NOTEBOOK_OUTPUT',adapter_zip)
print('SAVE_AS_KAGGLE_NOTEBOOK_OUTPUT',result_zip)
print('After Save Version: KAGGLE_OUTPUT_PRESERVED')